In [2]:
from pathlib import Path

import pandas as pd

# Carpeta con los CSV generados por `transform`
CSV_DIR = Path("../data/csv")

METRICS = ["views", "estimatedMinutesWatched"]

# Correspondencia entre los valores de creatorContentType y nuestro formato.
# OJO: son los valores que recuerdo, no verificados. Comprueba la salida de 5.1
# y edita este diccionario si los nombres que aparecen son otros.
MAPEO_OFICIAL = {
    "videoOnDemand": "long",
    "shorts": "short",
    "liveStream": "live",
    "posts": "posts",
    "creatorContentTypeUnspecified": "sin_tipo",
}

# Fecha en la que, según recuerdo (no verificado), el límite de un Short pasó de 60 a 180 s
CAMBIO_LIMITE_SHORTS = pd.Timestamp("2024-10-15", tz="UTC")

oficial = pd.read_csv(CSV_DIR / "analytics_channel_content_type.csv")
perf = pd.read_csv(CSV_DIR / "video_performance.csv")
channel_daily = pd.read_csv(CSV_DIR / "analytics_channel_daily.csv")

# --- 5.1 Informe oficial tal cual y comprobación de cobertura ---------------
print("== 5.1 Informe oficial (creatorContentType) ==")
print(oficial[["creatorContentType", *METRICS]].to_string(index=False))

sin_mapeo = sorted(set(oficial["creatorContentType"]) - set(MAPEO_OFICIAL))
if sin_mapeo:
    print(f"\nAVISO: valores sin correspondencia en MAPEO_OFICIAL: {sin_mapeo}")

print("\nVistas: suma del informe oficial vs total diario del canal")
print(f"  Informe oficial: {oficial['views'].sum():,.0f}")
print(f"  Canal (daily):   {channel_daily['views'].sum():,.0f}")

# --- 5.2 Comparación por formato --------------------------------------------
oficial["format"] = oficial["creatorContentType"].map(MAPEO_OFICIAL).fillna("otros_oficial")
oficial_por_formato = oficial.groupby("format")[METRICS].sum()

es_publico = perf["privacy_status"] == "public"
nuestro_todos = perf.groupby("format")[METRICS].sum()
nuestro_publicos = perf[es_publico].groupby("format")[METRICS].sum()

for metrica in METRICS:
    tabla = pd.DataFrame(
        {
            "oficial": oficial_por_formato[metrica],
            "nuestro_todos": nuestro_todos[metrica],
            "nuestro_publicos": nuestro_publicos[metrica],
        }
    ).fillna(0)
    tabla.loc["TOTAL"] = tabla.sum()

    # Evita dividir por cero si algún formato no tiene datos oficiales
    base = tabla["oficial"].where(tabla["oficial"] > 0)
    tabla["brecha_todos"] = tabla["oficial"] - tabla["nuestro_todos"]
    tabla["brecha_todos_%"] = tabla["brecha_todos"] / base * 100
    tabla["cuota_oficial_%"] = tabla["oficial"] / tabla.loc["TOTAL", "oficial"] * 100
    tabla["cuota_publicos_%"] = (
        tabla["nuestro_publicos"] / tabla.loc["TOTAL", "nuestro_publicos"] * 100
    )

    print(f"\n== 5.2 {metrica}: oficial vs nuestra clasificación ==")
    print(tabla.round(1).to_string())

# --- 5.3 Peso de los 7 públicos 'long' dentro del límite de un Short --------
perf["published_at"] = pd.to_datetime(perf["published_at"], utc=True)
limite = perf["published_at"].ge(CAMBIO_LIMITE_SHORTS).map({True: 180, False: 60})
dudosos = es_publico & (perf["format"] == "long") & (perf["duration_seconds"] <= limite)

vistas_dudosas = perf.loc[dudosos, "views"].sum()
vistas_long_pub = perf.loc[es_publico & (perf["format"] == "long"), "views"].sum()
vistas_short_pub = perf.loc[es_publico & (perf["format"] == "short"), "views"].sum()

print(f"\n== 5.3 Públicos 'long' con duración dentro del límite de un Short: {int(dudosos.sum())} ==")
print(f"Vistas de esos vídeos:           {vistas_dudosas:,.0f}")
print(f"  como % de las vistas 'long':   {vistas_dudosas / vistas_long_pub:.2%}")
print(f"  como % de las vistas 'short':  {vistas_dudosas / vistas_short_pub:.2%}")

== 5.1 Informe oficial (creatorContentType) ==
           creatorContentType   views  estimatedMinutesWatched
                videoOnDemand 2321129                 10052464
                       shorts 1725401                   862918
creatorContentTypeUnspecified   78745                   117061
                        posts     369                       34
                   liveStream      35                       12

Vistas: suma del informe oficial vs total diario del canal
  Informe oficial: 4,125,679
  Canal (daily):   4,126,269

== 5.2 views: oficial vs nuestra clasificación ==
          oficial  nuestro_todos  nuestro_publicos  brecha_todos  brecha_todos_%  cuota_oficial_%  cuota_publicos_%
format                                                                                                             
live           35        64252.0           64252.0      -64217.0       -183477.1              0.0               1.6
long      2321129      2267012.0         2245735.0       5